In [1]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.corpus import wordnet
import spacy
import os
from collections import deque
import json
import rarfile
import chardet
import re
import math

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Unaiza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

# Preprocessing

In [2]:
def load_stopwords(file):
    with open(file, "r") as f:
        stopwords = f.read().splitlines()
        return set(stopwords)

In [3]:
def casefolding(text):
    return text.lower()

In [4]:
def tokenize(text):
    """
    Normally, word_tokenize would be enough, but I noticed that it wasn’t handling 
    '-' and '/' properly in the abstracts. So, I had to manually replace them with spaces 
    before tokenizing to make sure the words are split correctly.
    """
    text = re.sub(r"[-/]", " ", text)  # replacing '-' and '/' with spaces
    tokens = word_tokenize(text)  # tokenizing the cleaned text
    return tokens

In [5]:
def remove_stopwords(tokens, stopwords):
    filtered_tokens = []
    for token in tokens:
        if token.isalnum():  # making sure the token is a proper word (alphanumeric) - (not punctuation, symbols, etc)
            if token not in stopwords: # keeping only words that are not in the stopwords list
                filtered_tokens.append(token)
    return filtered_tokens

In [6]:
# def lemmatize_tokens(tokens, lemmatizer):
#     tagged_tokens = pos_tag(tokens)
#     return [lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in tagged_tokens]

nlp = spacy.load("en_core_web_sm")

def lemmatize_tokens(tokens):
    text = " ".join(tokens)  # convert list of tokens to a string
    doc = nlp(text)
    return [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]

In [7]:
def preprocess(text, stopwords):
    text = casefolding(text) # 1. converting text to lowercase
    tokens = tokenize(text) # 2. tokenising the text
    tokens = remove_stopwords(tokens, stopwords) # 3. removing stopwords from text
    tokens = lemmatize_tokens(tokens) # 4. lemmatise tokens
    return tokens

In [8]:
def read_abstracts(folder):
    """
    Reading all abstract files from the given folder and storing their content in a dictionary.
    Since abstracts can have different encodings, I’m using `chardet` to detect the correct encoding for each file.
    First, I read the file in binary mode to detect its encoding.
    Then, I open it using the detected encoding and store the text in a dictionary.
    """
    abstracts = {}
    
    for filename in os.listdir(folder):  
        file = os.path.join(folder, filename)  

        if os.path.isfile(file):  
            with open(file, "rb") as f:  # reading as binary to detect encoding
                raw_data = f.read()
                result = chardet.detect(raw_data)
                encoding = result['encoding']  # detecting encoding

            try:
                # opening the file with the detected encoding
                with open(file, "r", encoding=encoding, errors="replace") as f:
                    abstracts[filename] = f.read()  # storing filename and file content in asbtracts disctionary
            except Exception as e:
                print(f"Error reading {filename}: {e}") 
    return abstracts

In [9]:
# lemmatizer = WordNetLemmatizer()

In [10]:
stopwords = load_stopwords(r"C:\Users\Unaiza\IR ASSIGNMENTS\Stopword-List.txt")

In [11]:
abstracts_folder = r"C:\Users\Unaiza\IR ASSIGNMENTS\abstracts"

In [12]:
abstracts = read_abstracts(abstracts_folder)

In [13]:
preprocessed_abstracts = {} # dictionary to store preprocessed abstracts
for doc, text in abstracts.items():
    preprocessed_abstracts[doc] = preprocess(text, stopwords)

In [14]:
print("Preprocessed Abstract [", list(preprocessed_abstracts.keys())[420], "]: ", # displaying a sample preprocessed file
      preprocessed_abstracts[list(preprocessed_abstracts.keys())[420]])

Preprocessed Abstract [ 74.txt ]:  ['genetic', 'algorithm', 'approach', 'partition', 'clustering', 'case', 'study', 'applicant', 'partition', 'cluster', 'cluster', 'divisive', 'analysis', 'genetic', 'algorithm', 'gower', 'measure', 'similarity', 'master', 'degree', 'information', 'technology', 'acquire', 'master', 'degree', 'common', 'practice', 'ensure', 'successful', 'life', 'good', 'career', 'path', 'especially', 'develop', 'country', 'master', 'degree', 'information', 'technology', 'popular', 'programme', 'prolific', 'number', 'application', 'student', 'work', 'main', 'objective', 'discover', 'number', 'cluster', 'applicant', 'characteristic', 'cluster', 'develop', 'genetic', 'algorithm', 'base', 'partition', 'cluster', 'program', 'achieve', 'incorporate', 'distance', 'matrix', 'application', 'divisive', 'analysis', 'gower', 'measure', 'similarity', 'genetic', 'algorithm', 'base', 'partition', 'cluster', 'program', 'develop', 'prove', 'superior', 'common', 'cluster', 'technique']


# Building Inverted Index

In [15]:
def build_inverted_index(preprocessed_abstracts):
    inverted_index = {}
    for doc, tokens in preprocessed_abstracts.items(): # iterating through each document and its preprocessed tokens 
        unique_tokens = set(tokens) # making sure there no duplicated tokens
        for token in unique_tokens: # iterating through each unique token 
            if token not in inverted_index: 
                inverted_index[token] = [] # if the token is not in the index already, initialise an empty list for it
            inverted_index[token].append(doc) # adding doc in the tokens list
    return inverted_index

In [16]:
inverted_index = build_inverted_index(preprocessed_abstracts)

In [17]:
with open(r"C:\Users\Unaiza\IR ASSIGNMENTS\inverted_index.json", "w") as f:
    json.dump(inverted_index, f) # saving the inverted index

In [18]:
print("Inverted Index of [", list(inverted_index.keys())[366], "]: ", # displaying a sample inverted index 
      inverted_index[list(inverted_index.keys())[366]])

Inverted Index of [ knowledge ]:  ['105.txt', '121.txt', '127.txt', '129.txt', '132.txt', '140.txt', '151.txt', '171.txt', '175.txt', '179.txt', '18.txt', '21.txt', '230.txt', '234.txt', '239.txt', '24.txt', '252.txt', '253.txt', '283.txt', '284.txt', '312.txt', '318.txt', '332.txt', '34.txt', '348.txt', '358.txt', '365.txt', '375.txt', '381.txt', '386.txt', '39.txt', '412.txt', '424.txt', '443.txt', '444.txt', '52.txt', '55.txt', '82.txt', '91.txt']


In [19]:
#print("Loading the entire Inverted Index: ", json.dumps(inverted_index, indent=4))

# Building Positional Index

In [20]:
def build_positional_index(preprocessed_abstracts):
    positional_index = {}
    for doc, tokens in preprocessed_abstracts.items(): # iterating through each document and its preprocessed tokens
        for pos, token in enumerate(tokens): # iterating through each token and its position in the doc
            if token not in positional_index:
                positional_index[token] = {} # if the token is not already in the index, create an empty dictionary for it
            if doc not in positional_index[token]:  
                positional_index[token][doc] = [] # if the doc is not already listed under the token, create an empty list for it
            positional_index[token][doc].append(pos) # storing the position of the token in the document
    return positional_index

In [21]:
positional_index = build_positional_index(preprocessed_abstracts)

In [22]:
with open(r"C:\Users\Unaiza\IR ASSIGNMENTS\positional_index.json", "w") as f:
    json.dump(positional_index, f) # saving the positional index 

In [23]:
print("Positional Index of [", list(positional_index.keys())[250], "]: ", # displaying a sample positional index 
      positional_index[list(positional_index.keys())[250]])

Positional Index of [ apply ]:  {'102.txt': [82], '109.txt': [51], '12.txt': [81], '131.txt': [81], '134.txt': [97], '145.txt': [76], '148.txt': [50], '150.txt': [41], '156.txt': [66], '160.txt': [67, 79], '17.txt': [69], '170.txt': [17], '171.txt': [96], '178.txt': [138], '180.txt': [83], '211.txt': [66], '214.txt': [8], '216.txt': [28], '221.txt': [53], '222.txt': [29], '225.txt': [45], '226.txt': [119], '229.txt': [24, 111], '236.txt': [59], '238.txt': [90], '239.txt': [65], '243.txt': [51, 58], '253.txt': [70], '255.txt': [40], '257.txt': [51, 76], '259.txt': [96], '262.txt': [99], '267.txt': [138], '268.txt': [107, 119], '27.txt': [91], '272.txt': [20], '275.txt': [59], '277.txt': [85], '278.txt': [50], '279.txt': [33], '280.txt': [24], '284.txt': [39], '286.txt': [57, 69], '293.txt': [52], '295.txt': [128], '298.txt': [150], '299.txt': [70, 93], '3.txt': [0], '301.txt': [49], '306.txt': [0, 26, 72], '310.txt': [87], '317.txt': [103], '318.txt': [110], '323.txt': [81], '325.txt': 

In [24]:
#print("Loading the entire Positional Index: ", json.dumps(positional_index, indent=4))

# Vector Space Model

### Term Feature Selection

In [25]:
def scale_tf(tf):
    return 1 + math.log10(tf) if tf > 0 else 0 # scaling 1 + log10(tf) for each term's frequency

In [26]:
def calculate_scaled_tf(preprocessed_abstracts):
    scaled_tf = {}  # scaled tf for each document
    for doc, tokens in preprocessed_abstracts.items():
        scaled_tf[doc] = {}
        raw_tf = {} # raw tf
        for token in tokens:
            if token in raw_tf:
                raw_tf[token] += 1
            else:
                raw_tf[token] = 1
        
        for token, count in raw_tf.items(): # scaling each tf
            scaled_tf[doc][token] = scale_tf(count)
    
    return scaled_tf

In [27]:
def calculate_idf(inverted_index, N):
    idf = {}  # idf for each term
    for term, docs in inverted_index.items(): 
        df = len(docs)  # calculate df for the term
        idf[term] = math.log(N / df)  # compute idf using the formula
    return idf  

In [28]:
def calculate_tf_idf(tf, idf):
    tf_idf = {}  # tfidf values for each term in each document
    for doc, term_freqs in tf.items():  
        tf_idf[doc] = {} 
        for term, freq in term_freqs.items():  
            tf_idf[doc][term] = freq * idf.get(term, 0.0)  # compute tfidf by multiplying tf & idf
    return tf_idf  

In [29]:
# build a vocabulary from the tf-idf dictionary (all unique words)
def build_vocab(tf_idf):
    vocab = set()  # empty set to store unique words
    for docs, term_weights in tf_idf.items():  
        vocab.update(term_weights.keys())  # add terms to the vocabulary set
    return sorted(vocab) 

In [30]:
def vectorize_documents(tf_idf, vocab): # converting tf-idf scores into vectors
    doc_vectors = {}  # store vectors for each document
    for docs, term_weights in tf_idf.items(): 
        vector = [term_weights.get(term, 0.0) for term in vocab]  # make a vector using the vocab order
        doc_vectors[docs] = vector  # save the vector
    return doc_vectors

### Cosine Similarity

In [31]:
def compute_vector_length(vector):
    return math.sqrt(sum(weight**2 for weight in vector.values()))

In [32]:
def cosine_similarity(query_vec, doc_vec):
    dot_product = sum(query_vec.get(term, 0) * doc_vec.get(term, 0) for term in query_vec) # dot product of query and document vectors
    query_length = compute_vector_length(query_vec) # calculate length of query vector
    doc_length = compute_vector_length(doc_vec) # calculate length of document vector
    if query_length == 0 or doc_length == 0: # if either vector has zero length, return 0
        return 0 # avoid division by zero
    return dot_product / (query_length * doc_length) # cosine similarity score

### Query Processing

In [33]:
def preprocess_query(query, stopwords):
    query_tokens = preprocess(query, stopwords) # applyig preprocessing steps to the query
    return query_tokens

In [34]:
def compute_scaled_query_tf(query_tokens):
    query_tf = {}  # raw tf for query
    for token in query_tokens: # counting raw tf
        if token in query_tf:
            query_tf[token] += 1
        else:
            query_tf[token] = 1
    scaled_query_tf = {token: scale_tf(count) for token, count in query_tf.items()} # scaling each tf
    
    return scaled_query_tf

In [35]:
def compute_query_tf_idf(query_tf, idf):
    query_tf_idf = {}  
    for term, tf_value in query_tf.items(): 
        query_tf_idf[term] = tf_value * idf.get(term, 0.0)  # multiply tf with idf value; if term not in idf, use 0.0 as default
    return query_tf_idf 

In [36]:
def vectorize_query(query_tf_idf, vocab):
    query_vector = [query_tf_idf.get(term, 0.0) for term in vocab] # creating a vector for the query using tf-idf scores; use 0.0 if term not in query

    return query_vector

In [37]:
def process_query(query, stopwords, idf, vocab):
    # preprocess the query
    query_tokens = preprocess_query(query, stopwords)
    
    # compute tf for the query
    query_tf = compute_scaled_query_tf(query_tokens)
    
    # compute tfidf for the query using the same idf from documents
    query_tf_idf = compute_query_tf_idf(query_tf, idf)
    
    # vectorize the query using the same vocabulary
    query_vector = vectorize_query(query_tf_idf, vocab)
    
    return query_vector

In [38]:
def retrieve_documents(query, stopwords, idf, vocab, doc_vectors_dict, alpha):
    query_vector_list = process_query(query, stopwords, idf, vocab) 
    query_vector_dict = {vocab[i]: query_vector_list[i] for i in range(len(vocab))}
    
    results = []
    for doc, doc_vector in doc_vectors_dict.items():
        # calculate cosine similarity between query vector and document vector
        score = cosine_similarity(query_vector_dict, doc_vector)
        
        # check if the document's similarity score meets or exceeds the threshold (alpha)
        if score >= alpha:
            results.append((doc, score))  # append the document and its score to the results list
    
    # sort the results in descending order of similarity score
    results.sort(key=lambda x: x[1], reverse=True)
    return results

In [39]:
def load_gold_standard_queries(file):
    gold_standard_queries = {}

    with open(file, "r") as f: # open the file containing the gold standard queries
        lines = f.readlines()

    current_query = None
    doc_ids = []

    for line in lines:
        line = line.strip() # strip leading/trailing spaces from the line
        if not line: # skip empty lines
            continue
        match = re.match(r"^\d+\)\s+(.+)$", line) # match the pattern for a query line (e.g., "1) query text")
        
        if match:# if the line matches the query pattern
            # if there was a previous query, save it with its doc ids
            if current_query and doc_ids:
                gold_standard_queries[current_query] = set(doc_ids)
            
            # update the current query to the new one (in lowercase)
            current_query = match.group(1).strip().lower()
            
            # reset the document IDs list for the new query
            doc_ids = []
        else:
            # extract document IDs from the line (separated by commas)
            ids = [int(x.strip()) for x in line.split(",") if x.strip()]
            doc_ids.extend(ids)
            
    if current_query and doc_ids:
        gold_standard_queries[current_query] = set(doc_ids)
        
    return gold_standard_queries

In [40]:
gold_standard_query_file = r"C:\Users\Unaiza\IR ASSIGNMENTS\Gold Query-VSM.txt"
gold_standard_queries = load_gold_standard_queries(gold_standard_query_file)

In [41]:
for query, docs in gold_standard_queries.items():
    print(f"Query: {query}")
    print(f"Docs: {docs}\n")

Query: deep
Docs: {267, 396, 397, 398, 273, 401, 21, 278, 279, 280, 281, 24, 405, 415, 421, 174, 175, 176, 177, 432, 325, 213, 345, 346, 347, 348, 352, 376, 358, 360, 362, 245, 246, 247, 374, 250, 380, 254}

Query: weak heuristic
Docs: {1, 257, 35, 101, 391, 361, 299, 172, 429, 174, 306, 435, 213, 93, 413}

Query: principle component analysis
Docs: {357, 134, 102, 426, 364, 45, 112, 434, 53, 310, 311, 315, 445}

Query: human interaction
Docs: {256, 255, 7, 383, 265, 10, 391, 395, 145, 273, 403, 21, 22, 23, 26, 30, 289, 162, 164, 426, 171, 428, 174, 436, 186, 187, 444, 191, 194, 203, 83, 345, 98, 101, 230, 369, 247, 249, 250, 127}

Query: supervised kernel k-means cluster
Docs: {264, 275, 280, 281, 158, 31, 291, 167, 427, 173, 430, 177, 53, 447, 334, 368, 241, 242, 243, 244, 245, 122, 123, 124, 125, 383}

Query: patients depression anxiety
Docs: {259, 263, 391, 400, 37, 40, 168, 433, 62, 447, 448, 72, 328, 332, 333, 80, 225, 355, 368}

Query: local global clusters
Docs: {257, 134, 136, 

# Evaluation

In [42]:
tf = calculate_scaled_tf(preprocessed_abstracts)
idf = calculate_idf(inverted_index, len(preprocessed_abstracts))
tf_idf = calculate_tf_idf(tf, idf)

In [43]:
vocab = build_vocab(tf_idf)
doc_vectors = vectorize_documents(tf_idf, vocab)

In [44]:
doc_vectors_dict = {
    doc: {vocab[i]: vector[i] for i in range(len(vocab))}
    for doc, vector in doc_vectors.items()
}

In [45]:
def convert_filenames_to_doc_ids(result_set):
    return {doc.replace(".txt", "") for doc in result_set}

In [46]:
def calculate_metrics(actual, expected):
    actual = set(actual)
    expected = set(expected)
    tp = len(actual & expected)
    fp = len(actual - expected)
    fn = len(expected - actual)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

In [47]:
overall_precision = []
overall_recall = []
overall_f1 = []
overall_map = []
overall_mar = []

In [48]:
alpha = 0.05

In [49]:
def calculate_average_precision(retrieved_docs, relevant_docs):
    retrieved_docs = list(retrieved_docs)
    relevant_docs = set(relevant_docs)
    hits = 0  # count of relevant docs found
    total_precision = 0  # sum of precision at each relevant doc position
    
    for rank, doc in enumerate(retrieved_docs, 1):
        if doc in relevant_docs:
            hits += 1  # increment hit count when relevant doc is found
            total_precision += hits / rank  # add precision at this rank
    # return average precision, or 0 if no relevant docs exist
    return total_precision / len(relevant_docs) if len(relevant_docs) > 0 else 0

In [50]:
def calculate_average_recall(retrieved_docs, relevant_docs):
    retrieved_docs = list(retrieved_docs)
    relevant_docs = set(relevant_docs)
    hits = 0  # count of relevant docs found
    total_recall = 0  # sum of recall at each relevant doc position
    
    for rank, doc in enumerate(retrieved_docs, 1):
        if doc in relevant_docs:
            hits += 1  # increment hit count when relevant doc is found
            total_recall += hits / len(relevant_docs)  # add recall at this rank
    
    # return average recall, or 0 if no relevant docs exist
    return total_recall / len(relevant_docs) if len(relevant_docs) > 0 else 0

In [51]:
for query, expected_docs in gold_standard_queries.items():
    retrieved_docs = retrieve_documents(query, stopwords, idf, vocab, 
                                        doc_vectors_dict, alpha)
    retrieved_doc_ids = {int(doc.replace('.txt', '')) for doc, _ in retrieved_docs}
    
    print("="*92)
    print(f"\nQuery: {query}")
    print("->> Expected:", sorted(expected_docs))
    print("->> Retrieved:", sorted(retrieved_doc_ids))
    
    precision, recall, f1 = calculate_metrics(retrieved_doc_ids, expected_docs)
    overall_precision.append(precision)
    overall_recall.append(recall)
    overall_f1.append(f1)
    
    # calculate Average Precision (AP) and Average Recall (AR) for the query
    ap = calculate_average_precision(retrieved_doc_ids, expected_docs)
    ar = calculate_average_recall(retrieved_doc_ids, expected_docs)
    overall_map.append(ap)
    overall_mar.append(ar)
    
    print(f"\n->> Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1:.2f}")
    print(f"->> Average Precision (AP): {ap:.2f}, Average Recall (AR): {ar:.2f}")
    print("="*92)


Query: deep
->> Expected: [21, 24, 174, 175, 176, 177, 213, 245, 246, 247, 250, 254, 267, 273, 278, 279, 280, 281, 325, 345, 346, 347, 348, 352, 358, 360, 362, 374, 376, 380, 396, 397, 398, 401, 405, 415, 421, 432]
->> Retrieved: [21, 23, 24, 174, 175, 176, 177, 213, 245, 246, 247, 250, 254, 258, 267, 272, 273, 278, 279, 280, 281, 325, 333, 345, 346, 347, 348, 352, 357, 358, 360, 362, 371, 373, 374, 375, 376, 380, 381, 382, 396, 397, 398, 401, 404, 405, 415, 421, 432, 444]

->> Precision: 0.76, Recall: 1.00, F1-score: 0.86
->> Average Precision (AP): 0.73, Average Recall (AR): 0.51

Query: weak heuristic
->> Expected: [1, 35, 93, 101, 172, 174, 213, 257, 299, 306, 361, 391, 413, 429, 435]
->> Retrieved: [1, 35, 93, 101, 172, 174, 213, 257, 306, 413, 429, 435]

->> Precision: 1.00, Recall: 0.80, F1-score: 0.89
->> Average Precision (AP): 0.80, Average Recall (AR): 0.35

Query: principle component analysis
->> Expected: [45, 53, 102, 112, 134, 310, 311, 315, 357, 364, 426, 434, 445]
->>

In [52]:
avg_precision = sum(overall_precision) / len(overall_precision)
avg_recall = sum(overall_recall) / len(overall_recall)
avg_f1 = sum(overall_f1) / len(overall_f1)

In [53]:
mean_ap = sum(overall_map) / len(overall_map) if len(overall_map) > 0 else 0
mean_ar = sum(overall_mar) / len(overall_mar) if len(overall_mar) > 0 else 0

In [54]:
print("\n" + "=" * 60)
print(f"{'Overall System Performance':^60}")
print("=" * 60)
print(f"{'Overall Precision:':<30} {avg_precision:.2f}")
print(f"{'Overall Recall:':<30} {avg_recall:.2f}")
print(f"{'Overall F1-Score:':<30} {avg_f1:.2f}")
print(f"{'Mean Average Precision:':<30} {mean_ap:.2f}")
print(f"{'Mean Average Recall:':<30} {mean_ar:.2f}")
print("=" * 60 + "\n")


                 Overall System Performance                 
Overall Precision:             0.84
Overall Recall:                0.74
Overall F1-Score:              0.73
Mean Average Precision:        0.64
Mean Average Recall:           0.37



In [55]:
import gradio as gr
import json

def search_documents(query):
    results = retrieve_documents(query, stopwords, idf, vocab, doc_vectors_dict, 
                                 alpha=0.05)
    
    if results:
        sorted_results = sorted(results, key=lambda x: x[1], reverse=True)
        result_data = [[doc, score] for doc, score in sorted_results]
    else:
        result_data = [["No relevant documents found.", ""]]
    
    return result_data

with gr.Blocks(title="Vector Space Information Retrieval System") as demo:
    gr.Markdown("# Vector Space Information Retrieval System")
    
    query_input = gr.Textbox(label="Enter Query", placeholder="e.g., principle component analysis")
    search_btn = gr.Button("Search")
    output_table = gr.DataFrame(headers=["Document", "Score"], interactive=False)  
    search_btn.click(search_documents, inputs=query_input, outputs=output_table)  
demo.launch()

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
